In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# ==========================================
# 1. INITIALIZE THE BRAIN (LLM)
# ==========================================
# We use ChatOllama to talk to your local model. 
# num_predict limits the response tokens for faster local execution.
llm = ChatOllama(model="qwen2.5:7b", temperature=0, num_predict=256)


#

In [ ]:
# 2. DEFINE THE TOOLS (The Hands)
# ==========================================
@tool
def read_local_file(filename: str) -> str:
    """Reads the contents of a local text file given its filename."""
    try:
        with open(filename, "r") as f:
            return {
    "content": f.read(),
    "status": "success"
}
    except FileNotFoundError:
        return f"Error: The file '{filename}' was not found."

# Bind our python function directly to the LLM framework
tools = [read_local_file]
llm_with_tools = llm.bind_tools(tools)

In [ ]:


# ==========================================
# 3. THE AGENT EXECUTION LOOP (The Heart)
# ==========================================
def run_agent(user_prompt: str):
    print(f"\n🚀 User Goal: '{user_prompt}'")
    
    # Initialize the short-term conversation memory history
    messages = [HumanMessage(content=user_prompt)]
    
    # We loop up to 5 times to let the agent think, act, and self-correct
    for step in range(5):
        print(f"\n--- [Step {step + 1}] Brain is thinking... ---")
        
        # Invoke the LLM with the complete history
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Check if the LLM brain decided it needs to use a tool
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                tool_id = tool_call["id"]
                
                print(f"🤖 Agent Decision: I need to use the tool '{tool_name}' with args: {tool_args}")
                
                # Execute the real Python function based on the LLM's request
                if tool_name == "read_local_file":
                    observation = read_local_file.invoke(input=tool_args)
                else:
                    observation = "Error: Tool not found."
                
                print(f"🛠️ Tool Output (Observation): {observation}")
                
                # Feed the tool's result back into memory so the LLM can inspect it
                messages.append(ToolMessage(content=str(observation), tool_call_id=tool_id))
        else:
            # If there are no tool calls, the LLM has reached its final answer
            print(f"\n🎯 Final Answer from Agent:\n{response.content}")
            return


In [ ]:

# ==========================================
# 4. RUNNING THE AGENT
# ==========================================
if __name__ == "__main__":
    # Create a dummy file for the agent to look at
    with open("todo.txt", "w") as f:
        f.write("1. Finish the ML math alignment module.\n2. Set up pgAdmin4 connection strings.\n3. Buy groceries.")
        
    # Ask the agent to perform a multi-step analytical task using your local file
    run_agent("Read my todo.txt file and count how many tasks I have left to do.")